In [4]:
# ============================================================
# 1. LOAD RESOLUTIONS + BUILD GOLDEN LIST
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from token_train import build_golden_list, weak_label_corpus

# ------------------------------------------------------------
# Resolutions
# ------------------------------------------------------------
df_res = pd.read_csv("../resolutions/ga_resolutions_1946_2019.csv")
df_res = df_res.rename(columns={"res_id2": "id"})

# Parse resolution date: DD Month YYYY
df_res["date_parsed"] = pd.to_datetime(
    df_res["date_c"],
    format="%d %B %Y",
    errors="coerce"
)

df_res["year"] = df_res["date_parsed"].dt.year

print("Resolutions:", len(df_res))
print("Date range:", df_res["date_parsed"].min(), "→", df_res["date_parsed"].max())
print("Year range:", df_res["year"].min(), "→", df_res["year"].max())


# ------------------------------------------------------------
# Instrument reference tables
# ------------------------------------------------------------
cols = ["title", "year", "alternative_name"]

# Wikipedia
df_wiki = pd.read_csv("../treaties/wiki-treaties_formatted.csv")
df_wiki = df_wiki.rename(
    columns={
        "name": "title",
        "cleaned_note": "alternative_name"
    }
)[cols]

# OHCHR
df_ohchr = pd.read_csv(
    "../ohchr_instruments/ohchr_instruments_detailed-instit.csv"
)
df_ohchr["year"] = pd.to_datetime(
    df_ohchr["adoption_date"],
    format="%d %B %Y",
    errors="coerce"
).dt.year
df_ohchr["alternative_name"] = ""
df_ohchr = df_ohchr[cols]

# UNO Treaties
df_uno = pd.read_csv("../treaties/UNO-Treaties.csv")
df_uno["year"] = pd.to_datetime(
    df_uno["date"],
    format="%d %B %Y",
    errors="coerce"
).dt.year
df_uno["alternative_name"] = ""
df_uno = df_uno[cols]

# UNESCO
df_unesco = pd.read_csv(
    "../unesco_instruments/UNESCO_legal_instruments_detail.csv"
)
df_unesco["year"] = pd.to_datetime(
    df_unesco["date"],
    format="%d %B %Y",
    errors="coerce"
).dt.year
df_unesco["alternative_name"] = ""
df_unesco = df_unesco[cols]

# Conventions / protocols / recommendations
df_conv_prot_rec = pd.read_csv(
    "../conv-prot-rec/conventions-protocols-recommendations.csv"
)
df_conv_prot_rec["alternative_name"] = ""
df_conv_prot_rec = df_conv_prot_rec[cols]


# ------------------------------------------------------------
# Combined instrument reference table
# ------------------------------------------------------------
df_instrument = pd.concat(
    [
        df_wiki,
        df_ohchr,
        df_uno,
        df_unesco,
        df_conv_prot_rec
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Golden list
# ------------------------------------------------------------
golden_list = build_golden_list(df_instrument)

# Remove duplicates while preserving order
golden_list = list(dict.fromkeys(
    x.strip()
    for x in golden_list
    if isinstance(x, str) and x.strip()
))

print("Reference rows:", len(df_instrument))
print("Golden-list names / aliases:", len(golden_list))

# Quick inspection
pd.Series(golden_list, name="instrument").head(20)

Resolutions: 17913
Date range: 1946-01-24 00:00:00 → 2019-09-16 00:00:00
Year range: 1946 → 2019
Reference rows: 1394
Golden-list names / aliases: 1414


0                                       Treaty of Paris
1                                  Treaty of Washington
2     Convention for the Preservation of Wild Animal...
3                                 Hay–Pauncefote Treaty
4                                        Boxer Protocol
5                               Anglo-Japanese Alliance
6                                 Treaty of Vereeniging
7             Cuban–American Treaty of Relations (1903)
8                                     Hay–Herrán Treaty
9                                    Hay–Herbert Treaty
10                             Hay–Bunau-Varilla Treaty
11                                 Treaty of Petrópolis
12             Southern African Customs Union Agreement
13                    International Sanitary Convention
14                Treaty of Peace and Friendship (1904)
15                                      Treaty of Lhasa
16                                 Treaty of Portsmouth
17                                     Treaty of

In [5]:
# ============================================================
# 2. TEMPORAL COVERAGE OF THE GOLDEN LIST
# ============================================================

# Run weak labeling on the full resolution corpus
known_docs, unknown_docs = weak_label_corpus(
    df_res,
    golden_list
)

print(f"Known docs:   {len(known_docs):,}")
print(f"Unknown docs: {len(unknown_docs):,}")


# ------------------------------------------------------------
# Collect instrument mentions + resolution year
# ------------------------------------------------------------
mentions = []

for doc in known_docs:
    doc_id = doc["doc_id"]

    # Get resolution year
    year = df_res.loc[df_res["id"] == doc_id, "year"]

    if year.empty or pd.isna(year.iloc[0]):
        continue

    year = int(year.iloc[0])

    for start, end, matched_text in doc["spans"]:
        mentions.append({
            "doc_id": doc_id,
            "year": year,
            "instrument": matched_text.strip(),
            "instrument_norm": matched_text.strip().casefold()
        })

df_mentions = pd.DataFrame(mentions)

print("Total matched mentions:", len(df_mentions))
print("Distinct matched names:", df_mentions["instrument_norm"].nunique())


# ------------------------------------------------------------
# Golden list as dataframe
# ------------------------------------------------------------
df_golden = pd.DataFrame({
    "instrument": golden_list
})

df_golden["instrument_norm"] = (
    df_golden["instrument"]
    .str.strip()
    .str.casefold()
)

# Defensive deduplication
df_golden = df_golden.drop_duplicates("instrument_norm").reset_index(drop=True)


# ------------------------------------------------------------
# First / last recognition year
# ------------------------------------------------------------
recognition_years = (
    df_mentions
    .groupby("instrument_norm")["year"]
    .agg(
        first_year="min",
        last_year="max",
        n_years="nunique",
        n_mentions="size"
    )
    .reset_index()
)

df_golden = df_golden.merge(
    recognition_years,
    on="instrument_norm",
    how="left"
)


# ------------------------------------------------------------
# Pre/post-2000 flags
# ------------------------------------------------------------
flags = (
    df_mentions
    .groupby("instrument_norm")["year"]
    .agg(
        has_pre_2000=lambda s: (s < 2000).any(),
        has_post_2000=lambda s: (s >= 2000).any()
    )
    .reset_index()
)

df_golden = df_golden.merge(
    flags,
    on="instrument_norm",
    how="left"
)

df_golden["has_pre_2000"] = df_golden["has_pre_2000"].fillna(False)
df_golden["has_post_2000"] = df_golden["has_post_2000"].fillna(False)

# ------------------------------------------------------------
# Classification
# ------------------------------------------------------------
df_golden["recognition_period"] = np.select(
    [
        (~df_golden["has_pre_2000"]) & (~df_golden["has_post_2000"]),
        ( df_golden["has_pre_2000"]) & (~df_golden["has_post_2000"]),
        (~df_golden["has_pre_2000"]) & ( df_golden["has_post_2000"]),
        ( df_golden["has_pre_2000"]) & ( df_golden["has_post_2000"]),
    ],
    [
        "never recognized",
        "pre-2000 only",
        "2000+ only",
        "both periods"
    ],
    default="unknown"
)


# ------------------------------------------------------------
# Main percentage
# ------------------------------------------------------------
n_golden = len(df_golden)

pct_only_post_2000 = (
    100
    * (df_golden["recognition_period"] == "2000+ only").sum()
    / n_golden
)

print(f"Golden-list names: {n_golden:,}")
print(f"Recognized only from 2000 onward: {pct_only_post_2000:.2f}%")

KeyboardInterrupt: 

In [ ]:
summary = (
    df_golden["recognition_period"]
    .value_counts()
    .rename_axis("recognition_period")
    .to_frame("n")
)

summary["pct"] = 100 * summary["n"] / len(df_golden)

summary

In [ ]:
# ============================================================
# 3. CUMULATIVE RECOGNITION CURVE
# ============================================================

years = np.arange(
    int(df_res["year"].min()),
    int(df_res["year"].max()) + 1
)

coverage = []

for year in years:
    recognized = (
        df_golden["first_year"].notna()
        & (df_golden["first_year"] <= year)
    ).sum()

    coverage.append(
        100 * recognized / len(df_golden)
    )

df_coverage = pd.DataFrame({
    "year": years,
    "pct_instruments_recognized": coverage
})


# Plot
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    df_coverage["year"],
    df_coverage["pct_instruments_recognized"],
    linewidth=2
)

ax.axvline(
    2000,
    linestyle="--",
    linewidth=1.5,
    label="2000"
)

pct_2000 = df_coverage.loc[
    df_coverage["year"] == 2000,
    "pct_instruments_recognized"
].iloc[0]

ax.scatter([2000], [pct_2000], zorder=3)

ax.annotate(
    f"{pct_2000:.1f}%",
    xy=(2000, pct_2000),
    xytext=(2005, pct_2000 + 3),
    arrowprops=dict(arrowstyle="->")
)

ax.set_xlabel("Year")
ax.set_ylabel("% of golden-list instruments recognized")
ax.set_title("Cumulative recognition of golden-list instruments")
ax.set_ylim(0, 100)
ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4. PRE-2000 / POST-2000 COMPOSITION
# ============================================================

order = [
    "pre-2000 only",
    "both periods",
    "2000+ only",
    "never recognized"
]

composition = (
    df_golden["recognition_period"]
    .value_counts()
    .reindex(order, fill_value=0)
)

composition_pct = 100 * composition / len(df_golden)

fig, ax = plt.subplots(figsize=(10, 3))

left = 0

for category in order:
    value = composition_pct[category]

    ax.barh(
        ["Golden list"],
        [value],
        left=left,
        label=category
    )

    if value > 4:
        ax.text(
            left + value / 2,
            0,
            f"{value:.1f}%",
            ha="center",
            va="center"
        )

    left += value

ax.set_xlim(0, 100)
ax.set_xlabel("% of golden list")
ax.set_title("When are golden-list instruments represented in resolutions?")
ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
annual_recognition = (
    df_mentions
    .groupby("year")["instrument_norm"]
    .nunique()
    .div(len(df_golden))
    .mul(100)
    .reset_index(name="pct_golden_recognized")
)

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    annual_recognition["year"],
    annual_recognition["pct_golden_recognized"],
    marker="o",
    markersize=3
)

ax.axvline(
    2000,
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel("Year")
ax.set_ylabel("% of golden list recognized that year")
ax.set_title("Annual representation of golden-list instruments")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()